In [ ]:
from jax import random
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import yaml as yml
import pycountry

from emu_renewal.inputs import BASE_PATH, DATA_PATH, get_who_targets, get_hosp_target, get_standard_priors
from emu_renewal.inputs import get_european_var_props, get_filtered_seroprev, get_worldbank_national_pop
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import store_outputs
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
# Standard inputs
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
proc_update_freq = 7
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2021, 4, 1)
seed_duration = 10
analysis_to_data_delay = 14
iterations = 1000

# Loaded inputs
country_inputs = yml.safe_load(open(BASE_PATH / "emu_renewal/inputs.yml", "r"))
priors = get_standard_priors()

In [ ]:
for country in ["Sweden"]:
    print(country)
    iso2 = pycountry.countries.lookup(country).alpha_2
    iso3 = pycountry.countries.lookup(country).alpha_3
    
    # Collate mobility data options
    g_mob = pd.read_csv(DATA_PATH / f"mobility/{iso2}_gmob_data.csv", index_col=0)
    g_mob.index = pd.to_datetime(g_mob.index)
    nonresi_g_mob = g_mob.loc[:, g_mob.columns!="residential"].mean(axis=1).rolling(7).mean().dropna()
    
    fb_mob = pd.read_csv(DATA_PATH / f"mobility/{iso3}_fbmob_data.csv", index_col=0)["0"]
    fb_mob.index = pd.to_datetime(fb_mob.index)
    fb_mob = 1.0 + fb_mob.rolling(7).mean().dropna()
    
    collated_mob = pd.DataFrame(
        {
            "google_nonresi_linear": nonresi_g_mob,
            "google_nonresi_square": nonresi_g_mob ** 2.0,
            "fb_linear": fb_mob,
            "fb_square": fb_mob ** 2.0,
        },
    )
    collated_mob["no_mob"] = 1.0

    for mob_analysis_type in collated_mob:
        print(mob_analysis_type)
    
        # Get targets
        cases_target, deaths_target, init_data = get_who_targets(country, analysis_start, analysis_end, init_duration, analysis_to_data_delay)
        hosp_target = get_hosp_target(country, analysis_start, analysis_end, analysis_to_data_delay)
        seroprev_target = get_filtered_seroprev(country, analysis_start, analysis_end)
        
        # Europe-specific wrangling of variants
        strains = ["eu", "alpha"]
        var_target = get_european_var_props(country, var_target_start_date, var_target_end_date, strains)
        alpha_seed_start = analysis_start + timedelta(country_inputs["alpha_seed_delays"][country])
        alpha_seed_times = [alpha_seed_start, alpha_seed_start + timedelta(seed_duration)]
        seed_times = [alpha_seed_times]
        
        # Collate targets
        seroprev_target_dict = {"seropos": StandardDispTarget(seroprev_target, weight=20.0)} if any(seroprev_target) else {}
        all_targets = {
            "weekly_cases": StandardDispTarget(cases_target, weight=20.0),
            "weekly_deaths": StandardDispTarget(deaths_target, weight=20.0),
            "prop_eu": StandardDispTarget(var_target, weight=20.0),
            "occupancy": StandardDispTarget(hosp_target, weight=20.0),
        } | seroprev_target_dict
        
        # Define model and fitter
        model = MultiStrainModel(
            get_worldbank_national_pop(iso3),
            analysis_start,
            analysis_end,
            proc_update_freq,
            CosineMultiCurve(),
            GammaDens(),
            init_duration,
            init_data,
            GammaDens(),
            GammaDens(),
            strains,
            "eu",
            seed_times,
            100.0,
            collated_mob[mob_analysis_type].dropna(),
        )
        
        # Run calibration
        calib = StandardCalib(model, priors, all_targets, proc_dispersion=dist.HalfNormal(0.5))
        analysis_time = datetime.now().strftime("%Y%m%d_%H%M")
        kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
        mcmc = infer.MCMC(kernel, num_chains=4, num_samples=iterations, num_warmup=iterations)
        mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])
        store_outputs(country, mob_analysis_type, analysis_time, model, calib, mcmc)